## AI Math Assistant with Langchain Tool Calling

In [1]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv
import re

In [2]:
load_dotenv()

True

In [3]:
#load the LLM
llm = ChatOpenAI(model="gpt-4", temperature=0.3, max_tokens=400)

In [4]:
response = llm.invoke("What is tool calling?")

In [5]:
print("\nResponse Content: ", response.content)


Response Content:  Tool calling typically refers to the process of invoking or using a specific tool, software, or application within a larger system or program. This can be done through various methods such as command line interfaces, APIs, or scripts. The specific process can vary depending on the tool and the system it is being used in.


In [6]:
# Function for adding numbers 
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]


    result = sum(numbers)
    return {"result": result}

In [7]:
add_numbers("10 20 30")

{'result': 60}

In [8]:
# Tool Class 
from langchain_core.tools import Tool
add_tool = Tool(
    name="AddTool",
    func=add_numbers,
    description="Adds a list of numbers and returns the result.")


In [9]:
print("tool object", add_tool)

tool object name='AddTool' description='Adds a list of numbers and returns the result.' func=<function add_numbers at 0x0000029C96E11090>


In [10]:
# Tool name
print("Tool Name:")
print(add_tool.name)

# Tool description
print("Tool Description:")
print(add_tool.description)

# Tool function
print("Tool Function:")
print(add_tool.invoke)

Tool Name:
AddTool
Tool Description:
Adds a list of numbers and returns the result.
Tool Function:
<bound method BaseTool.invoke of Tool(name='AddTool', description='Adds a list of numbers and returns the result.', func=<function add_numbers at 0x0000029C96E11090>)>


In [11]:
print("Calling Tool Function:")
test_input = "10 20 40 b c"
print(add_tool.invoke(test_input))

Calling Tool Function:
{'result': 70}


In [12]:
# @tool operator  -- Recommned way to create tools

@tool
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    # Use regular expression to extract all numbers from the input 
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    
    result = sum(numbers)
    return {"result": result}


In [13]:
print("Name: \n", add_numbers.name)
print("Description: \n", add_numbers.description) 
print("Args: \n", add_numbers.args) 

Name: 
 add_numbers
Description: 
 Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

Parameters:
- inputs (str): 
string, it should contain numbers that can be extracted and summed.

Returns:
- dict: A dictionary with a single key "result" containing the sum of the numbers.

Example Input (Dictionary):
{"numbers": [10, 20, 30]}

Example Input (String):
"Add the numbers 10, 20, and 30."

Example Output:
{"result": 60}
Args: 
 {'inputs': {'title': 'Inputs', 'type': 'string'}}


In [14]:
test_input = "what is the sum between 10, 20 and 30 " 
print(add_numbers.invoke(test_input))

{'result': 60}


In [15]:
# Comparing the two approaches
print("Tool Constructor Approach:")

print(f"Has Schema: {hasattr(add_tool, 'args_schema')}")
print("\n")

print("@tool Decorator Approach:")


print(f"Has Schema: {hasattr(add_numbers, 'args_schema')}")
print(f"Args Schema Info: {add_numbers.args}")

Tool Constructor Approach:
Has Schema: True


@tool Decorator Approach:
Has Schema: True
Args Schema Info: {'inputs': {'title': 'Inputs', 'type': 'string'}}


In [16]:
# Adding tool with two inputs : first add and second boolean as an input
from typing import List

@tool
def add_numbers_with_options(numbers: List[float], absolute: bool = False) -> float:
    """
    Adds a list of numbers provided as input.

    Parameters:
    - numbers (List[float]): A list of numbers to be summed.
    - absolute (bool): If True, use the absolute values of the numbers before summing.

    Returns:
    - float: The total sum of the numbers.
    """
    
    if absolute:
        numbers = [abs(num) for num in numbers]
    
    return sum(numbers)


In [17]:
print(f"Args Schema Info: {add_numbers_with_options.args}")
print(f"Args Schema Info: {add_numbers.args}")

Args Schema Info: {'numbers': {'items': {'type': 'number'}, 'title': 'Numbers', 'type': 'array'}, 'absolute': {'default': False, 'title': 'Absolute', 'type': 'boolean'}}
Args Schema Info: {'inputs': {'title': 'Inputs', 'type': 'string'}}


In [18]:
print(add_numbers_with_options.invoke({"numbers":[-2.1,-5.1,-3.0],"absolute":True}))
print(add_numbers_with_options.invoke({"numbers":[-2.1,-5.1,-3.0],"absolute":False}))

10.2
-10.2


In [ ]:
from typing import Dict, Union

@tool
def sum_numbers_with_complex_output(inputs: str) -> Dict[str, Union[float, str]]:
    """
    Extracts and sums all integers and decimal numbers from the input string.

    Parameters:
    - inputs (str): A string that may contain numeric values.

    Returns:
    - dict: A dictionary with the key "result". If numbers are found, the value is their sum (float). 
            If no numbers are found or an error occurs, the value is a corresponding message (str).

    Example Input:
    "Add 10, 20.5, and -3."

    Example Output:
    {"result": 27.5}
    """
    
    matches = re.findall(r'-?\d+(?:\.\d+)?', inputs)
    
    if not matches:
        return {"result": "No numbers found in the input."}
    
    try:
        #numbers = [abs(float(num)) for num in matches]
        numbers = [float(num) for num in matches]
        total = sum(numbers)
        return {"result": total}
    
    except Exception as e:
        return {"result": f"Error processing numbers: {str(e)}"}

In [20]:
@tool
def sum_numbers_from_text(inputs: str) -> float:
    """
    Adds a list of numbers provided in the input string.
    
    Args:
        text: A string containing numbers that should be extracted and summed.
        
    Returns:
        The sum of all numbers found in the input.
    """
    
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    result = sum(numbers)
    
    return result

In [21]:
from langchain.agents import create_agent
agent = create_agent(llm, tools=[add_tool])

In [35]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Add the numbers 10, 20, 30, two"}]})

In [36]:
final_answer = result["messages"][-1].content
print(final_answer)

The sum of the numbers 10, 20, 30, and 2 is 62.


## create_react_agent
As LangChain's AgentExecutor is being deprecated, create_react_agent from LangGraph provides a more flexible and powerful alternative for building AI agents. This function creates a graph-based agent that works with chat models and supports tool-calling functionality.

## Key parameters of create_react_agent
1. model
The language model that powers the agent's reasoning.
Must support tool calling for full functionality.

2. tools
A list of tools the agent can use to perform actions.
Can be LangChain tools, Python functions with @tool decorator, or a ToolNode instance
Each tool should have a name, description, and implementation

3. prompt (optional):
Customizes the instructions given to the LLM
Can be:
A string (converted to a SystemMessage)
A SystemMessage object
A function that transforms the state
A Runnable that processes the state
and other parameters. To see more parameters, see docs.

## How it works
Unlike the legacy AgentExecutor, which used a fixed loop structure, create_react_agent creates a graph with these key nodes:

1. Agent Node: Calls the LLM with the message history
2. Tools Node: Executes any tool calls from the LLM's response
3. Continue/End Nodes: Manage the workflow based on whether tool calls are present

The graph follows this process:
1. User message enters the graph
2. LLM generates a response, potentially with tool calls
3. If tool calls exist, they're executed and their results are added to the message history
4. The updated messages are sent back to the LLM
5. This loop continues until the LLM responds without tool calls
6. The final state with all messages is returned

In [45]:
from langgraph.prebuilt import create_react_agent

agent_exec = create_react_agent(model=llm, tools=[sum_numbers_with_complex_output])
msgs = agent_exec.invoke({"messages": [("human", "Add the numbers -10, 20, -30")]})

C:\Users\akans\AppData\Local\Temp\ipykernel_26852\165224076.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_exec = create_react_agent(model=llm, tools=[sum_numbers_with_complex_output])


In [46]:
print(msgs["messages"][-1].content)

The sum of the numbers -10, 20, -30 is -20.


In [47]:
@tool
def new_subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and performs subtraction sequentially, starting with the first number.

    This function is designed to handle input in string format, where numbers may be separated by spaces, 
    commas, or other delimiters. It parses the input string, extracts numeric values, and calculates 
    the result by subtracting each subsequent number from the first. inputs[0]-inputs[1]-inputs[2]

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string can include spaces, commas, or other 
      delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Usage:
    - Input: "100, 20, 10"
    - Output: {"result": 70}

    Limitations:
    - The function does not handle cases where numbers are formatted with decimals or other non-integer representations.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number
    result = numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}

In [48]:
# Multiplication Tool
@tool
def multiply_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates their product.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the product of the numbers.

    Example Input:
    "2, 3, 4"

    Example Output:
    {"result": 24}

    Notes:
    - If no numbers are found, the result defaults to 1 (neutral element for multiplication).
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]
    print(numbers)

    # If no numbers are found, return 1
    if not numbers:
        return {"result": 1}

    # Calculate the product of the numbers
    result = 1
    for num in numbers:
        result *= num
        print(num)

    return {"result": result}

In [49]:
# Division Tool
@tool
def divide_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates the result of dividing the first number 
    by the subsequent numbers in sequence.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the quotient.

    Example Input:
    "100, 5, 2"

    Example Output:
    {"result": 10.0}

    Notes:
    - If no numbers are found, the result defaults to 0.
    - Division by zero will raise an error.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]


    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Calculate the result of dividing the first number by subsequent numbers
    result = numbers[0]
    for num in numbers[1:]:
        result /= num

    return {"result": result}

In [52]:
tools = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers]
# Create the agent with all tools
math_agent = create_react_agent(
    model=llm,
    tools=tools,
    # Optional: Add a system message to guide the agent's behavior
    prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)
print("agent",math_agent)

agent <langgraph.graph.state.CompiledStateGraph object at 0x0000029C99820070>


C:\Users\akans\AppData\Local\Temp\ipykernel_26852\504588550.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  math_agent = create_react_agent(


In [53]:
# Test Cases
test_cases = [
    {
        "query": "Add 100, 20, and 10.",
        "expected": {"result": 130},
        "description": "Testing addition tool with sequential addition."
    },
    {
        "query": "Multiply 2, 3, and 4.",
        "expected": {"result": 24},
        "description": "Testing multiplication tool for a list of numbers."
    },
    {
        "query": "Divide 100 by 5 and then by 2.",
        "expected": {"result": 10.0},
        "description": "Testing division tool with sequential division."
    },
    {
        "query": "Subtract 50 from 20.",
        "expected": {"result": -30},
        "description": "Testing subtraction tool with negative results."
    }

]

In [ ]:
correct_tasks = []

# Execute Test Cases
for index, test in enumerate(test_cases, start=1):
    query = test["query"]
    expected_result = test["expected"]["result"]  # Extract just the value
    # Example:
    # test["expected"] = {"result": 30}
    # expected_result = 30
    

    
